#### Importação de bibliotecas

In [1]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

#### Dataset LEVIR-CD

In [2]:
class LEVIRCDDataset(Dataset):
    def __init__(self, images_dir, transform=None):
        # caminhos para imagens pré-mudança (A), pós-mudança (B) e máscaras
        self.images_dir_a = os.path.join(images_dir, 'A')  # antes
        self.images_dir_b = os.path.join(images_dir, 'B')  # depois
        self.masks_dir = os.path.join(images_dir, 'label') # máscara
        self.transform = transform

        # lista ordenada de arquivos de imagem em cada diretório
        self.images_a = sorted(f for f in os.listdir(self.images_dir_a) 
                               if f.endswith(('.png', '.jpg', '.jpeg')))
        self.images_b = sorted(f for f in os.listdir(self.images_dir_b) 
                               if f.endswith(('.png', '.jpg', '.jpeg')))
        self.masks = sorted(f for f in os.listdir(self.masks_dir) 
                            if f.endswith(('.png', '.jpg', '.jpeg')))

    def __len__(self):
        return len(self.images_a)

    def __getitem__(self, idx):
        # monta caminhos completos para o par de imagens e a máscara correspondente
        img_path_a = os.path.join(self.images_dir_a, self.images_a[idx])
        img_path_b = os.path.join(self.images_dir_b, self.images_b[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])
                
        # leitura e pré-processamento das imagens: BGR → RGB → float normalizado [0, 1]
        image_a = cv2.imread(img_path_a, cv2.IMREAD_COLOR)
        image_a = cv2.cvtColor(image_a, cv2.COLOR_BGR2RGB)
        image_a = image_a.astype(np.float32) / 255.0

        image_b = cv2.imread(img_path_b, cv2.IMREAD_COLOR)
        image_b = cv2.cvtColor(image_b, cv2.COLOR_BGR2RGB)
        image_b = image_b.astype(np.float32) / 255.0

        # leitura da máscara binária em escala de cinza e normalização para [0, 1]        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = mask.astype(np.float32) / 255.0

        # aplica augmentações sincronizadas nas duas imagens e na máscara
        if self.transform:
            augmented = self.transform(image=image_a, image1=image_b, mask=mask)
            image_a = augmented['image']
            image_b = augmented['image1']
            mask = augmented['mask']
                
        # adiciona dimensão de canal à máscara: (H, W) → (1, H, W)
        mask = mask.unsqueeze(0)

        return image_a, image_b, mask

# transformações albumentations
# treino: redimensionamento + augmentações geométricas aleatórias + conversão para tensor
train_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2(),
], additional_targets={'image1': 'image'})  # sincroniza augmentação entre imagem pré e pós

# validação: apenas redimensionamento e conversão para tensor (sem augmentação)
val_transform = A.Compose([
    A.Resize(height=256, width=256),
    ToTensorV2(),
], additional_targets={'image1': 'image'})


#### Criação dos datasets e DataLoaders

In [3]:
# caminho raiz do dataset LEVIR-CD e tamanho do batch
root = os.path.join(os.getcwd(), "LEVIR_CD")
batch_size = 16

LEVIR_train_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'train'),
    transform = train_transform
)

LEVIR_val_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'val'),
    transform = val_transform
)

LEVIR_test_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'test'),
    transform = val_transform # sem augmentação, igual à validação
)

# criação dos DataLoaders para treino, validação e teste
# shuffle = True no treino para embaralhar os batches e melhorar a generalização
LEVIR_train_loader = DataLoader(LEVIR_train_dataset, batch_size=batch_size, shuffle=True)
LEVIR_val_loader = DataLoader(LEVIR_val_dataset, batch_size=batch_size, shuffle=False)
LEVIR_test_loader = DataLoader(LEVIR_test_dataset, batch_size=batch_size, shuffle=False)

#### Treinamento e validação

A cada epoch o modelo treina no conjunto de treino, avalia no de validação e registra as métricas (Loss, IoU, Dice, Precision, Recall, Accuracy). O melhor modelo é salvo automaticamente com base no IoU de validação.

<small>

| Métrica | Indica |
|---|---|
| `Loss` | erro geral do modelo — **quanto menor, melhor** |
| `IoU` | sobreposição entre máscara predita e real — métrica principal |
| `Dice` | equilíbrio entre precisão e recall — similar ao IoU |
| `Precision` | dos pixels classificados como mudança, quantos realmente eram |
| `Recall` | das mudanças reais, quantas foram detectadas |
| `Accuracy` | pixels corretos no geral — pode ser enganosa em dados desbalanceados |

<small>

In [ ]:
import torch.optim as optim

def calculate_metrics(outputs, labels, threshold=0.5):
    # converte logits em probabilidades e aplica threshold para obter máscara binária
    probs = torch.sigmoid(outputs)
    predictions = (probs > threshold).float()
    
    # achata para vetor 1D para facilitar os cálculos
    predictions = predictions.view(-1)
    labels = labels.float().view(-1)

    # verdadeiros/falsos positivos e negativos
    tp = torch.sum(predictions * labels)
    tn = torch.sum((1 - predictions) * (1 - labels))
    fp = torch.sum(predictions * (1 - labels))
    fn = torch.sum((1 - predictions) * labels)

    # métricas — 1e-6 evita divisão por zero
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (tp + fn + 1e-6)
    dice      = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou       = tp / (tp + fp + fn + 1e-6)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + 1e-6)

    return (accuracy.item(), precision.item(), recall.item(), dice.item(), iou.item())

# loop de treinamento e validação
num_epochs = 50

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device='cuda'):
    model.to(device)
    best_iou = 0.0
    # histórico das métricas
    history = {
        'train_loss': [], 'train_iou': [], 'train_acc': [],
        'train_precision': [], 'train_recall': [], 'train_dice': [],
        'val_loss': [],   'val_iou': [],   'val_acc': [],
        'val_precision': [],   'val_recall': [],   'val_dice': [],
    }

    for epoch in range(num_epochs):
        #fase do teste
        model.train()
        running_train_loss = 0.0
        train_iou_sum = train_acc_sum = train_precision_sum = 0
        train_recall_sum = train_dice_sum = train_samples = 0
        
        for inputs1, inputs2, labels in train_loader:
            inputs1, inputs2, labels = inputs1.to(device), inputs2.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs1, inputs2)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # evita explosão de gradientes
            optimizer.step()
            
            # acumula loss e métricas ponderadas pelo tamanho do batch
            n = inputs1.size(0)
            running_train_loss += loss.item() * n
            acc, precision, recall, dice, iou = calculate_metrics(outputs, labels)
            train_acc_sum += acc * n
            train_precision_sum += precision * n
            train_recall_sum += recall * n
            train_dice_sum += dice * n
            train_iou_sum += iou * n
            train_samples += n
        
        # médias ponderadas do epoch de treino
        epoch_train_loss        = running_train_loss    / train_samples
        epoch_train_iou         = train_iou_sum         / train_samples
        epoch_train_acc         = train_acc_sum         / train_samples
        epoch_train_precision   = train_precision_sum   / train_samples
        epoch_train_recall      = train_recall_sum      / train_samples
        epoch_train_dice        = train_dice_sum        / train_samples
        
        # registra no histórico
        history['train_loss'].append(epoch_train_loss)
        history['train_iou'].append(epoch_train_iou)
        history['train_acc'].append(epoch_train_acc)
        history['train_precision'].append(epoch_train_precision)
        history['train_recall'].append(epoch_train_recall)
        history['train_dice'].append(epoch_train_dice)

        # fase de validação
        model.eval()
        running_val_loss = 0.0
        val_iou_sum = val_acc_sum = val_precision_sum   = 0
        val_recall_sum = val_dice_sum = val_samples     = 0
        
        with torch.no_grad(): # desativa gradientes para economizar memória
            for inputs1, inputs2, labels in val_loader:
                inputs1, inputs2, labels = inputs1.to(device), inputs2.to(device), labels.to(device)
                outputs = model(inputs1, inputs2)
                loss = criterion(outputs, labels)
                
                n = inputs1.size(0)
                running_val_loss += loss.item() * n
                acc, precision, recall, dice, iou = calculate_metrics(outputs, labels)
                val_acc_sum += acc * n
                val_precision_sum += precision * n
                val_recall_sum += recall * n                
                val_dice_sum += dice * n
                val_iou_sum += iou * n
                val_samples += n
        
        # médias ponderadas do epoch de validação
        epoch_val_loss      = running_val_loss  / val_samples
        epoch_val_iou       = val_iou_sum       / val_samples
        epoch_val_acc       = val_acc_sum       / val_samples
        epoch_val_precision = val_precision_sum / val_samples
        epoch_val_recall    = val_recall_sum    / val_samples
        epoch_val_dice      = val_dice_sum      / val_samples

        history['val_loss'].append(epoch_val_loss)
        history['val_iou'].append(epoch_val_iou)
        history['val_acc'].append(epoch_val_acc)
        history['val_precision'].append(epoch_val_precision)
        history['val_recall'].append(epoch_val_recall)
        history['val_dice'].append(epoch_val_dice)
        
        # log do epoch
        log = (f'Epoch {epoch+1}/{num_epochs} || '
              f'Treino Loss: {epoch_train_loss:.4f} / Val Loss: {epoch_val_loss:.4f} || '
              f'Treino IoU: {epoch_train_iou:.4f} / Val IoU: {epoch_val_iou:.4f} || '
              f'Treino Accuracy: {epoch_train_acc:.4f} / Val Accuracy: {epoch_val_acc:.4f} || '
              f'Treino Precision: {epoch_train_precision:.4f} / Val Precision: {epoch_val_precision:.4f} || '
              f'Treino Recall: {epoch_train_recall:.4f} / Val Recall: {epoch_val_recall:.4f} || '
              f'Treino Dice: {epoch_train_dice:.4f} / Val Dice: {epoch_val_dice:.4f}')
        
        print(f'--- LOG DE TREINAMENTO ---')
        print(log)
        with open('train_log.txt', 'a') as f:
            f.write(log + '\n')

        # checkpoint ─ salva o melhor modelo com base no IoU de validação
        if epoch_val_iou > best_iou:
            best_iou = epoch_val_iou

            torch.save({
                'epoch':                    epoch + 1,
                'model_state_dict':         model.state_dict(),
                'optimizer_state_dict':     optimizer.state_dict(),
                'iou':                      epoch_val_iou,
                'loss':                     epoch_val_loss
            }, 'best_model.pt')
            print(f' -> best_model.pt salvo! IoU: {best_iou:.6f}')

        # salva checkpoint do último epoch
        torch.save({
            'epoch':                        epoch + 1,
            'model_state_dict':             model.state_dict(),
            'optimizer_state_dict':         optimizer.state_dict(),
            'best_iou':                     best_iou,
            'iou':                          epoch_val_iou,
            'loss':                         epoch_val_loss,
            'train_loss':                   epoch_train_loss
        }, "last_checkpoint.pt")

    return history

# configuração e início do treinamento
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SiameseUNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss() # combina sigmoid + BCE, numericamente mais estável
optimizer = optim.Adam(model.parameters(), lr=0.0001)

history = train_model(model, LEVIR_train_loader, LEVIR_val_loader, criterion, optimizer, num_epochs, device=device)

# carrega o melhor modelo salvo para avaliação
checkpoint = torch.load('best_model.pt', map_location = device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval();